# Predicting Hotel Booking Cancellations
MSIS 502 Final Team Project

Team members: Eva Maheshwari, Hilton Nguyen, Kayla Pham, Serena Mei, William Somat​

<br>

###Project Goal <br>
Roughly one in three hotel bookings is cancelled before the guest ever arrives. Every cancellation leaves a room that could have been sold to someone else, and by the time the hotel finds out it is usually too late to resell it at a good rate.

<br>

This project asks a single question: can we tell, at the moment a booking is made, whether it is likely to be cancelled?

<br>

Who this is for: Our decision-maker is a hotel manager that has the capacity to decide how much to overbook, which channels to invest in, and when to require a deposit. A model that flags high-risk bookings early lets them hold back fewer rooms, target retention effort, and price more confidently in peak season.

<br>

What we do: We clean the data, explore which booking characteristics are associated with cancellation, then build and compare two predictive models: an interpretable classification tree and a random forest.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
hotels = pd.read_csv('hotel_bookings.csv')

print("Rows and columns:", hotels.shape)
hotels.head()

FileNotFoundError: [Errno 2] No such file or directory: 'hotel_bookings.csv'

## The Dataset

We're using the Hotel Booking Demand dataset from Kaggle
(https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand), which comes
from a 2019 paper by Antonio, Almeida and Nunes.

It has 119,390 bookings and 32 columns, covering two real hotels in Portugal (one city hotel and one resort) for arrivals between July 2015 and August 2017. All data is extracted from hotels' Property Management System(PMS) SQL database. Each row is one booking, and it includes both stays that happened and stays that got cancelled.

The column we're trying to predict is `is_canceled`, which is 1 if the booking was
cancelled and 0 if the guest showed up.

The other columns give us a lot to work with. There's timing information like how far
in advance the booking was made and how many nights it was for, details about the party
size, which channel the booking came through, whether the guest had stayed or cancelled
before, the nightly price, and what the guest asked for (special requests, parking).

# <br>

One thing worth knowing: `adr` stands for average daily rate, which is basically the
nightly price in euros.

## Data Cleaning and Preprocessing

Before modeling, we inspect the data for missing values and for values that are not realistic.

In [ ]:
# Missing values
print(hotels.isna().sum().sort_values(ascending=False).head(6))

In [ ]:
# Check for impossible values
print(hotels['adr'].describe())
print()
print(hotels['adults'].describe())

In [ ]:
# Data cleaning
hotels_clean = hotels.copy()

hotels_clean = hotels_clean.drop('company', axis=1)
hotels_clean['agent'] = hotels_clean['agent'].fillna(0)
hotels_clean['country'] = hotels_clean['country'].fillna('Unknown')
hotels_clean['children'] = hotels_clean['children'].fillna(0)

hotels_clean = hotels_clean[(hotels_clean['adr'] >= 0) & (hotels_clean['adr'] < 1000)]
hotels_clean = hotels_clean[hotels_clean['adults'] > 0]

print("Missing values remaining:", hotels_clean.isna().sum().sum())
print("Shape after cleaning:", hotels_clean.shape)
print("Rows removed:", hotels.shape[0] - hotels_clean.shape[0])

Cleaning removed 405 rows out of 119,390. That is about 0.34% of the data. We retain retain 99.7% of the original bookings.

### What we found

Four columns had missing values, and we handled each one differently depending on how bad the problem was.

`company` was missing in 112,593 rows, which is about 94% of the data so we have decided to dropped remove it as a whole.

`agent` was missing in 16,340 rows (14%). A blank agent ID probably just means the booking didn't come through a travel agent, so we filled those with 0 instead of treating them as errors.

`country` was only missing in 488 rows, less than half a percent. We filled those with "Unknown" rather than deleting the rows since everything else about those bookings was fine.

`children` was missing in just 4 rows. We filled those with 0.

<br>

We also found some values that don't make sense. The nightly rate (`adr`) goes as low as -6.38 and as high as 5,400, even though 75% of bookings are under 126. You can't have a negative room rate, and one booking at 5,400 is almost certainly a mistake, so we kept only bookings between 0 and 1,000. We also found bookings with 0 adults,
which can't be a real reservation, so we decided to drop the unrealistic ones.

In [ ]:
print(hotels_clean['is_canceled'].value_counts())
print()
print((hotels_clean['is_canceled'].value_counts(normalize=True) * 100).round(1))

About 37% of bookings get cancelled. That's a decent split to work with because both outcomes happen often enough for the model to learn what each one looks like. If almost every booking went ahead and hardly any were cancelled, a model could just guess "not cancelled" every single time and still look accurate, without actually being useful.

## 4. Exploratory Data Analysis

We now look for booking characteristics associated with cancellation. Each figure below is
followed by what it shows and why it matters to a revenue manager.

### 4.1 Does the property type matter?

In [ ]:
rate_by_hotel = hotels_clean.groupby('hotel')['is_canceled'].mean() * 100
print(rate_by_hotel.round(2))
print()
print(hotels_clean['hotel'].value_counts())

sns.barplot(x=rate_by_hotel.index,
            y=rate_by_hotel.values,
            hue=rate_by_hotel.index,
            palette={
        'City Hotel': '#4C78A8',
        'Resort Hotel': '#F2A541'},
            )

plt.xlabel('Hotel Type')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Hotel Type')
plt.show()

The City Hotel gets cancelled a lot more often than the Resort Hotel.

41.8% versus 27.8%.

The City Hotel is also the bigger of the two, with 78,939 bookings out of 118,985, so it has both the higher cancellation rate and more bookings at stake.

The two hotels shouldn't be treated the same way. Since the City Hotel loses a bigger share of its bookings, it needs to plan for more cancellations than the resort does.

### 4.2 How far ahead was the booking made?

In [ ]:
print(hotels_clean.groupby('is_canceled')['lead_time'].describe().round(1))

sns.boxplot(data=hotels_clean,
            x='is_canceled',
            y='lead_time',)
plt.xlabel('Cancelled (0 = No, 1 = Yes)')
plt.ylabel('Lead Time (days before arrival)')
plt.title('Booking Lead Time by Cancellation Outcome')
plt.show()

Bookings that end up getting cancelled are made much further in advance. For cancelled bookings, the middle value is 113 days before arrival. For bookings that actually happen, it's 45 days. The two boxes in the chart barely overlap, which means lead time separates the two groups better than any other number we looked at.

For example; Someone booking four months ahead is much less committed than someone booking six weeks out as plans change. And since the hotel knows the lead time the moment a booking comes in, this is something they can act on right away rather than finding out too late.

### 4.3 Does it matter where the booking came from?

In [ ]:
seg = hotels_clean.groupby('market_segment')['is_canceled'].agg(['mean', 'count'])
seg['mean'] = (seg['mean'] * 100).round(2)
seg = seg.sort_values('mean', ascending=False)
print(seg)

# Leave out the Undefined segment since there are only 2 bookings
seg_plot = seg[seg.index != 'Undefined']

sns.barplot(x=seg_plot.index,
            y=seg_plot['mean'],
            hue=seg_plot.index,
            palette='Blues_r',
            legend=False)
plt.xlabel('Booking Channel')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Booking Channel')
plt.xticks(rotation=45, ha='right')
plt.show()

Where the booking comes from makes a big difference. Group bookings get cancelled 61.1% of the time, while bookings made directly with the hotel only get cancelled 15.4% of the time, and corporate bookings 18.8%. Online travel agents like Booking.com or Expedia sit in the middle at 36.8%, but they bring in way more bookings than anyone else (56,221), so even at that rate they end up producing the highest total number of cancellations.

A direct booking is worth more to the hotel than a group booking at the
same price, because it's about four times more likely to actually happen. So it makes sense for the hotel to spend money getting people to book directly.

<br>

One thing we noticed: There's an "Undefined" category with only 2 bookings, and both were cancelled. We left it out of the chart so
it doesn't mislead, but it's still in the table above.

###4.4 Does the Stays Type matter?


In [ ]:
import numpy as np

conditions = [
    (hotels_clean['stays_in_weekend_nights'] > 0) &
    (hotels_clean['stays_in_week_nights'] == 0),

    (hotels_clean['stays_in_week_nights'] > 0) &
    (hotels_clean['stays_in_weekend_nights'] == 0),

    (hotels_clean['stays_in_weekend_nights'] > 0) &
    (hotels_clean['stays_in_week_nights'] > 0)
]

stay_types = [
    'Weekend Only',
    'Weekday Only',
    'Mixed Stay'
]

hotels_clean['stay_type'] = np.select(
    conditions,
    stay_types,
    default='No Overnight Stay'
)
stay_data = hotels_clean[
    hotels_clean['stay_type'] != 'No Overnight Stay'
]

rate_by_stay_type = (
    stay_data.groupby('stay_type')['is_canceled'].mean() * 100
)

order = ['Weekday Only', 'Weekend Only', 'Mixed Stay']
rate_by_stay_type = rate_by_stay_type.reindex(order)

print(rate_by_stay_type.round(2))
print()
print(stay_data['stay_type'].value_counts())

deposit_color = {
    'Weekday Only': '#4C78A8',
    'Weekend Only': '#F2A541',
    'Mixed Stay': '#59A14F'
}

# Create bar chart
sns.barplot(
    x=rate_by_stay_type.index,
    y=rate_by_stay_type.values,
    hue=rate_by_stay_type.index,
    palette=deposit_color,
    legend=False
)

plt.xlabel('Stay Type')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Weekday and Weekend Stay Type')
plt.show()

Weekend-only bookings have the lowest cancellation rate at 29.58%, while weekday-only and mixed-stay bookings have much higher rates of 40.77% and 40.47%, respectively.

This suggests that weekend-only guests may have more definite travel plans, while weekday and mixed-stay customers represent greater cancellation risk. The hotel could use targeted reminders, deposits, or limited cancellation policies for weekday and mixed-stay reservations. These differences could also help the hotel improve overbooking and room-inventory decisions, although additional factors such as customer type and lead time should also be considered.

### 4.5 Do deposits prevent cancellation?

In [ ]:
dep = hotels_clean.groupby('deposit_type')['is_canceled'].agg(['mean', 'count'])
dep['mean'] = (dep['mean'] * 100).round(2)
print(dep)

deposit_color = {
    'No Deposit': '#4C78A8',
    'Non Refund': '#F2A541',
    'Refundable': '#59A14F'
}

sns.barplot(x=dep.index,
            y=dep['mean'],
            hue=dep.index,
            palette= deposit_color,
            legend=False,
               )
plt.xlabel('Deposit Type')
plt.ylabel('Cancellation Rate (%)')
plt.title('Cancellation Rate by Deposit Type')
plt.show()

Bookings with a non-refundable deposit were marked as cancelled 99.4% of the time, compared to 28.4% for bookings with no deposit. But the
rate on its own is misleading, and the more useful way to look at this is by volume.

Non-refundable bookings are only 12.3% of all bookings (14,586 out of 118,985). No Deposit bookings are 87.6%. When you count actual cancellations, No Deposit bookings make up 67.1% of them (29,585 out of 44,114) while non-refundable bookings are only 32.9%. So the group
with the scary-looking rate is the smaller problem, and the group with the ordinary-looking rate is where most of the lost bookings actually come from.

This matters for what the hotel should do. Chasing the 99.4% number would mean focusing effort on a small factor of the business. If the goal is to reduce total cancellations, the No Deposit group is where the volume is.

As for why the non-refundable rate is so high, we can't tell for certain from this data. The dataset records what the deposit terms were but not who set them or why. We did check whether these might be guests who paid and never showed up without formally cancelling, but the records show 14,459 marked "Canceled" against only 34 "No-Show," so that's not it.

Our best guess is that the hotel asks for non-refundable terms on bookings it already considers risky, which would mean the deposit is a reaction to the risk rather than the cause of it. We want to be clear that this is a **hypothesis** and not something we can prove here.

### 4.6 What else is linked to cancellation?

In [ ]:
numeric_cols = ['is_canceled', 'lead_time', 'adults', 'children', 'babies',
                'is_repeated_guest', 'previous_cancellations',
                'previous_bookings_not_canceled', 'booking_changes',
                'days_in_waiting_list', 'adr',
                'required_car_parking_spaces', 'total_of_special_requests']

corr = hotels_clean[numeric_cols].corr()
print(corr['is_canceled'].sort_values(ascending=False).round(3))

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Booking Features')
plt.show()

Lead time has the strongest connection to cancellation (+0.33), which lines up with what the boxplot from section 4.2 already showed us. Going the other way, the strongest negative connections are the number of special requests (-0.23) and whether the guest asked for a parking space (-0.21).

A negative number here just means the two things move in opposite directions. The more special requests a guest makes, the less likely they are to cancel. That's the interesting part. Guests who ask the hotel for things seem more committed to actually showing up.

None of these connections are especially strong. The biggest one is 0.33, which means longer lead times do tend to come with more cancellations, but it's a general trend rather than a reliable rule. Plenty of bookings made far in advance are still honored.

That tells us no single column can predict cancellation by itself. We need a model that weighs several of them together, which is what we will build in the upcoming section using a decision tree.

### Let's take a look at the two strongest negative signals

In [ ]:
# Unpacking the two strongest negative signals
print("Cancellation rate by number of special requests (%):")
print((hotels_clean.groupby('total_of_special_requests')['is_canceled'].mean() * 100).round(1))
print()
print("Cancellation rate by parking spaces requested (%):")
print((hotels_clean.groupby('required_car_parking_spaces')['is_canceled'].mean() * 100).round(1))
print()
print("Cancellation rate by repeat-guest status (%):")
print((hotels_clean.groupby('is_repeated_guest')['is_canceled'].mean() * 100).round(1))

From the table, bookings where the guest asked for nothing extra get cancelled 47.8% of the time, dropping to 22.0% with one request and 10.7% with four. Repeat guests cancel 14.7% versus 37.8% for first-timers. So guests who ask for
things, or who have stayed before, are noticeably more likely to show up.

Parking is a strange case. Out of 7,405 bookings that requested a space, zero were cancelled. That felt too perfect, and our guess is the column gets filled in at check-in rather than at booking, which would explain it. If so we  think that it shouldn't be using it to predict anything, since the hotel wouldn't know it yet. We flag this in our limitations section.

Unlike lead time, these are things the hotel can influence. Prompting guests to add a request while booking costs nothing, and the repeat guest gap is a good argument for a loyalty program.

### 4.7 Is there a seasonal pattern?

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

monthly = hotels_clean.groupby('arrival_date_month').agg(
    avg_adr=('adr', 'mean'),
    cancel_rate=('is_canceled', 'mean')
).reindex(month_order)
monthly['avg_adr'] = monthly['avg_adr'].round(2)
monthly['cancel_rate'] = (monthly['cancel_rate'] * 100).round(2)
print(monthly)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(monthly.index, monthly['avg_adr'], marker='o', color='tab:blue')
ax1.set_xlabel('Arrival Month')
ax1.set_ylabel('Average Daily Rate (EUR)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
plt.xticks(rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['cancel_rate'], marker='s', color='tab:red')
ax2.set_ylabel('Cancellation Rate (%)', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Average Daily Rate and Cancellation Rate by Arrival Month')
plt.tight_layout()
plt.show()

Room prices swing a lot through the year. August is the most expensive month at 140.39 euros a night, and January is the cheapest at 70.52. That's almost exactly double.

Cancellations are highest in June (41.5%) and April (40.8%), and lowest in November (31.3%) and January (30.5%).

One pattern we can see is the months when an empty room costs the hotel the most are also the months when guests are most likely to cancel. The hotel shouldn't plan for cancellations the same way all year. A room sitting empty in August costs about twice what it does in January, so the same cancellation rate hurts a lot more in summer.

## 5. Predictive Modelling
We now build a model that predicts, from information available **at the time of booking**, whether a reservation will be cancelled.

### 5.1 Two columns we had to leave out

There are two columns in this dataset that would have invalidated our model if we'd used them.

`reservation_status` tells you how the booking ended up: Check-Out, Canceled, or No-Show.

`reservation_status_date` is the date that outcome was recorded.

If we included either one, our model would basically be perfect. But it would be
meaningless, because the model wouldn't be predicting anything, just reading back a value that already contains the outcome. More importantly, the hotel doesn't know either of these things when the booking is first made, which is exactly when they need the prediction.

So we left both out of our predictor list below. The rule we used for picking columns was simple: Would the hotel actually know this at the moment someone books? If not, it can't go in the model.

### 5.2 Choosing predictors

We select features that are known at booking time and supported by the exploratory analysis above. Four of our columns hold text rather than numbers (hotel, market segment, deposit type, and customer type). The decision tree model we build in the next section can only work with numbers, so we use `pd.get_dummies()` to turn each text column into a set of yes/no columns.

In [ ]:
numeric_predictors = ['lead_time', 'total_of_special_requests',
                      'required_car_parking_spaces', 'booking_changes',
                      'previous_cancellations', 'is_repeated_guest',
                      'adr', 'adults', 'stays_in_week_nights',
                      'days_in_waiting_list']

categorical_predictors = ['hotel', 'market_segment', 'deposit_type', 'customer_type']

hotels_X = pd.get_dummies(
    hotels_clean[numeric_predictors + categorical_predictors],
    columns=categorical_predictors,
    drop_first=True
)
hotels_y = hotels_clean['is_canceled']

print("Predictor columns after encoding:", hotels_X.shape[1])
print(list(hotels_X.columns))

### 5.3 Partitioning the data

We split the bookings into two groups: 70% to train the model on, and 30% that we set aside and don't let it see.

By checking it against the 30% it's never seen, we get a fair sense of how it would do on real bookings in the future.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    hotels_X,
    hotels_y,
    test_size=0.3,
    random_state=2
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

### 5.4 Model 1: Classification tree

We start by limiting the tree to a maximum depth of 3, meaning it can only ask three questions before making a prediction. A deeper tree would be more accurate, but it would also be much harder to read. Keeping it smaller and more interpretable means the hotel can actually see the logic and judge whether it makes sense.

In [ ]:
tree_model = DecisionTreeClassifier(max_depth=3, random_state=20)
tree_model.fit(X_train, y_train)

print("Tree depth:", tree_model.get_depth())
print("Number of terminal leaves:", tree_model.get_n_leaves())

In [ ]:
plt.figure(figsize=(22, 10))

plot_tree(
    tree_model,
    feature_names=X_train.columns,
    class_names=['Not Canceled', 'Canceled'],
    filled=True,
    impurity=False,
    fontsize=9
)

plt.show()

**Reading the tree**

The very first split is on whether the booking has a non-refundable deposit. That means out of everything available, the algorithm found this was the most useful question to ask first. It matches what we saw earlier in section 4.4, and everything on that side of the tree gets predicted as a cancellation.

For the much bigger group of bookings without a non-refundable deposit, the tree asks about lead time next, then about whether the guest has cancelled before. Bookings made within about two weeks of the stay are predicted to go ahead. Bookings made further out, by guests who have cancelled in the past, are predicted to cancel.

Only three columns are doing real work here: deposit type, lead time, and previous cancellations. That keeps it readable, but it also limits what the tree can pick up on. The largest group of bookings are the ones made more than two weeks ahead by guests with no cancellation history. It contains 35,976 bookings that went through and 17,088 that were cancelled.

The model goes with the majority and predicts they all show up, which means it's wrong for about a third of them. That's a lot of our data sitting in a prediction the tree isn't confident about, which is why we will build a second model in the upcoming section.

### 5.5 Evaluating against a baseline

Accuracy on its own doesn't mean much. Since 63% of bookings weren't cancelled, a "model" that just guessed "not cancelled" every single time would still be right 63% of the time, without learning anything at all.

So 63% is the number our model has to beat.

In [ ]:
train_pred = tree_model.predict(X_train)
test_pred = tree_model.predict(X_test)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Training accuracy:", round(train_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))

# Baseline: always predict "not cancelled"
baseline_pred = [0] * len(y_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print("Baseline accuracy (always predict Not Cancelled):", round(baseline_accuracy, 4))

print("Improvement over baseline:", round(test_accuracy - baseline_accuracy, 4))

Model 1 gets 76.8% on the test data, compared to 63.2% if we'd just guessed "not cancelled" every time. So it's about 13 points better than doing nothing.

The training and test scores came out almost the same (76.6% and 76.8%), which is a good sign. It means the tree isn't just memorizing the bookings it trained on and then falling apart on new ones. Capping the depth at 3 is what kept that from happening.

The other thing worth pointing out is what that 63% baseline actually does. It only reaches that score by never flagging a single cancellation, so it would be no help to the hotel at all. Model 1 catches some of them, which is what we actually need it to do.

### 5.6 Model 2: Random forest

A random forest builds a lot of trees instead of just one, each on a random sample of the data, and then has them vote on the answer. The idea is that averaging over many trees reduces the mistakes any single tree would make on its own.

The downside is that you can't draw it. With one tree we could look at the diagram and follow the logic, but a forest of 100 trees has no single picture that explains it. We're trading away some of that clarity to see if we can get better accuracy.

In [ ]:
forest_model = RandomForestClassifier(n_estimators=100, random_state=20)
forest_model.fit(X_train, y_train)

forest_train_pred = forest_model.predict(X_train)
forest_test_pred = forest_model.predict(X_test)

print("Random forest training accuracy:", round(accuracy_score(y_train, forest_train_pred), 4))
print("Random forest test accuracy:", round(accuracy_score(y_test, forest_test_pred), 4))

The forest gets 84.3% on the test data, compared to 76.8% for Model 1 and 63.2% for just guessing. So it does predict better.

It scored 98.7% on the data it trained on and 84.3% on the data it hadn't seen. That's a big drop. Basically the forest learned the training bookings really well and then didn't do nearly as well on new ones. Model 1 didn't have that problem, since its two scores were 76.6% and 76.8%, almost the same.

We thought this was interesting because a random forest is supposed to handle this better than a single tree, and ours still showed a bigger gap than Model 1 did. It also shows why the test score is the one that counts. The test score
tells us how it does on bookings it's never seen, and that's the situation the hotel would actually be using it in. If we had reported 98.7% and the hotel planned around that, they would be disappointed.

The trade-off: The forest is more accurate, but Model 1 is the one you can
explain. If a hotel staff member asks why a booking got flagged as risky, Model 1 tree can be explained in about thirty seconds. With the forest, the honest answer is that 100 trees voted and most of them said cancel, which isn't a satisfying explanation for anyone.

Our recommendation is to use the forest to predict which bookings are risky, since that's the number the hotel would plan around and accuracy matters most there. But we'd suggest using Model 1 tree as the explanation. When someone asks what actually drives cancellations, the answer is deposit type, lead time, and whether the guest has cancelled before, and the tree shows that clearly.

### Which features does the forest rely on?

In [ ]:
importances = pd.Series(forest_model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(8).round(4))

Lead time comes out on top (0.275), then price/adr (0.234), then whether there was a non-refundable deposit (0.126) and the number of special requests (0.063). The forest spreads its attention across more columns than Model 1 did, which makes sense since it has way more splits to work with.

Price is the interesting one here. Model 1 never used it at all, not once.Model 1 only got three questions to ask, so it spent them on the most useful questions and never got around to price. The forest had far more room and found that price actually matters quite a bit. That lines up with our seasonality section, where the expensive months also had some of the highest cancellation rates.

### 5.7 What kind of mistakes does the model make?

Accuracy hides the difference between the two ways a model can be wrong, and those two errors
cost the hotel very different amounts.

In [ ]:
cm = confusion_matrix(y_test, forest_test_pred)
print("Confusion matrix:")
print(cm)
print()

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Not Canceled', 'Predicted: Canceled'],
            yticklabels=['Actual: Not Canceled', 'Actual: Canceled'])
plt.title('Random Forest Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

print(classification_report(y_test, forest_test_pred,
                            target_names=['Not Canceled', 'Canceled'], digits=3))

Out of 13,151 bookings in the test set that really were cancelled, the forest caught 9,807 and missed 3,344. So it finds about three quarters of them. And out of everything it flagged as a cancellation, roughly 8 out of 10 actually were.

The 3,344 are bookings that did get cancelled, but Model 2 predicted they'd be fine. So the hotel plans around a guest who was never going to arrive. By the time they find out, it's too late to sell the room to anyone else and it sits empty for the night.

The 2,245 bookings are the opposite. These bookings actually went through, but Model 2 predicted a cancellation. If the hotel had acted on that and sold the room to someone else, the original guest shows up to find no room available. Now the hotel is paying to put them up at another property and dealing with a guest who probably won't book with them again.

We think which mistake you would rather make depends on how full the hotel is. On a quiet night in January, an empty room isn't a big deal, so you'd want the model to be aggressive about flagging cancellations. On a sold-out weekend in August, turning a guest away is expensive and embarrassing, so you'd want it to be more careful. The hotel could adjust how cautious the model is depending on the season rather than leaving it on one setting all year.

## 6. Business Insights and Recommendations

### What we found

**Lead time is the most useful warning sign.** Bookings that got cancelled were made a median of 113 days before the stay, versus 45 days for the ones where the guest actually showed up. Out of all the number columns, this is the one where cancelled and non-cancelled bookings look most different from each other.

<br>

**Where the booking comes from matters a lot.** Direct bookings get cancelled 15.4% of the time. Group bookings get cancelled 61.1% of the time. Online travel agents sit in the middle at 36.7%, but they bring in far more bookings than anyone else, so they account for 46.8% of all cancellations by themselves. That's nearly half.

<br>

**Guests who ask for anything at all tend to show up.** Bookings with no special requests get cancelled 47.8% of the time. With one request that drops to 22.0%. Interestingly it doesn't keep falling much after that as two requests is 22.1%, basically the same as one. So the jump happens between asking for nothing and asking for something, not between asking for a little and asking for a lot. Repeat guests also cancel far less, 14.7% versus 37.8% for first-timers.

<br>

**Peak season is exposed on both sides.** Summer rooms cost roughly double winter ones, 140.39 euros in August against 70.52 in January, and the expensive months also cancel more, averaging 39.1% against 31.8% for the cheapest months. The hotel is most likely to lose a room in exactly the months where losing one costs the most.

<br>

**Both models beat guessing, but they trade off against each other.** Model 2 predicts better, 84.3% against Model 1's 76.8%, but it's a black box and it leaned heavily on memorizing the training data. Model 1 gets most of the way there on three columns anyone can follow. Both clear the 63.2% you'd get by guessing, so the real question isn't whether a model helps but which kind
of help the hotel needs.

<br>

**The loudest number is the least useful one.** Non-refundable bookings cancel 99.4% of the time, but they're 12.3% of the business, and we can't tell whether the deposit caused the cancellation or just marked a booking the hotel already doubted. The cancellations that actually matter are the two thirds coming from ordinary No Deposit bookings at 28.4%.

<br>

### What we'd recommend

**Set cancellation rates by booking source and season, not one number for everything.** How a guest books makes a big difference. Bookings made directly with the hotel cancel 15.4% of the time, while group bookings cancel 61.1%. Time of year matters too, from 30.5% in January
up to 41.5% in June. Planning around one average rate means the hotel is too cautious with the reliable bookings and not cautious enough with the risky ones.

<br>

**Spend more on getting guests to book directly.** Bookings made straight with the hotel cancel 15.4% of the time. Bookings that come through online travel sites like Booking.com or Expedia cancel 36.7% of the time. With the assumption that the hotel pays commission, a direct booking is definitely cheaper to get and much more likely to turn into an actual stay.

<br>

**Plan staffing around who will actually arrive, not how many bookings exist.** About 37% of all bookings get cancelled, and that rate shifts by month and by property. If housekeeping and front desk schedules are built off the raw booking count, the hotel might be staffing more people than it needs.

<br>

**Leave the deposit policy alone until it's been tested.** Bookings with a non-refundable deposit cancel 99.4% of the time, which looks alarming, but we can't tell from the data whether the deposit caused that or whether the hotel asked for deposits on bookings it already thought were risky. The way to find out is to apply non-refundable terms to a random set of ordinary bookings and see whether their cancellation rate changes.

<br>

### Who this is useful for

**Revenue managers** get the most out of it. They're the ones deciding how many extra rooms to sell on top of what's already booked, and what to charge for them. Knowing which specific bookings are likely to fall through helps with both of those calls.

**Marketing** can use the booking source comparison. Knowing that direct bookings cancel at 15.4% versus 36.7% through online travel sites changes how you'd value each channel when deciding where to spend.

**Operations** can plan staffing off expected arrivals rather than raw booking counts, since roughly 37% of bookings won't turn into guests.

<br>

### Data we wish we had

**What each cancellation actually cost.** The data records that a booking was cancelled, but not what happened next. Was the room resold? At what price? Without that, our model has no way to tell the difference between a cancellation the hotel absorbed easily and one that left a room empty on a sold-out night. Cost data would let us build something that optimizes for profit rather than getting right predictions.

**Which travel site the booking came through.** The data has one category called "Online TA" covering online travel agents, but it doesn't say which one. Since that single category produces 46.8% of all cancellations, breaking it down by individual site would show the hotel whether the problem is spread evenly or concentrated in one platform.

**Whether the hotel ever followed up with the guest.** Long lead times are our strongest warning sign, and the obvious response is to reach out before the stay. But nothing in the data records whether a confirmation email or reminder was ever sent, so we can't test whether
following up makes any difference.

<br>

### Limitations

- This is two hotels in Portugal between 2015 and 2017, all before the pandemic. Cancellation behavior has almost certainly changed since, and we wouldn't assume these numbers transfer to other markets or other kinds of property.

- We're not sure when the parking column gets filled in. If it's recorded at check-in rather than at booking, the hotel wouldn't know it when a booking comes in. We checked how much it affects us: Model 1 never used the column at all, and removing it from Model 2 only drops accuracy from 84.3% to 83.9%. So it isn't messing with our findings, but it's something the hotel would need to confirm before actually running the model, since a column that's empty at booking time is no
use for predicting anything.

- Our cleaning involved judgment calls. We kept 1,804 bookings with a nightly rate of exactly 0, which could be complimentary stays, staff bookings, or errors in the data. We chose to keep them since a rate of zero is at least possible, unlike a negative one.

- When we cleaned the data, we removed any booking with a nightly rate of 1,000 euros or more. That number was a judgment call on our part. It
turned out that it did not really matter as the most expensive booking in the whole dataset was 5,400 euros, and the next one down was 510. Since nothing sits in between, we would basically have removed exactly the same booking whether we had drawn the line at 600, 1,000, or 2,000.

- When we measured accuracy, we counted both kinds of mistake the same way. But as we showed in section 5.7, missing a cancellation leaves a room empty, while wrongly predicting one could mean a guest arrives to find no room available. Those aren't equally bad, and which is worse
depends on how full the hotel is. A model built around what each mistake costs would be more useful than one built around accuracy.

<br>

### What we'd do differently or next

**Get more recent data.** Ours stops in 2017, before the pandemic. Travel habits changed a lot after that, and it would be worth checking whether the patterns we found still hold.

**Build a model that thinks about cost, not just accuracy.** Right now our model tries to get as many predictions right as possible, treating every mistake the same. But an empty room and a guest arriving to find no room available cost the hotel very different amounts. A better version would be to try to save the hotel the most money rather than get the most answers right.

**Try predicting the nightly price too.** We only predicted whether a booking gets cancelled. The nightly rate is another column we could predict, using a regression model instead of a tree. If the hotel could estimate both how much a booking is worth and how likely it is to
fall through, they could decide which bookings to protect and which to overbook against.

<br>

### How we used AI

We used AI throughout the project and treated it as a guide as we generate our ideas. We let AI structure our brainstorming sessions to an easy to read and visualized process as we build this project.

It was most useful for structure and for catching things we'd have missed as it can recheck our math, logic, and code.

We pushed back quite a bit on AI as sometimes it gives us the wrong assumption/answer. It likes to pull from general knowledge about how things usually work rather than sticking to what the data actually shows, so we had to verify the numbers ourselves each time and make our final checks.

The lesson we took from it is that AI definitely can speed up the mechanical work a lot, but checking its numbers/logic/assumptions against our own output was necessary every time, and the judgment about
what a result actually means still had to come from us.